In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
!pip install optuna mlflow dagshub lightgbm seaborn

In [ ]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import dagshub
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import clone
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [ ]:
# Initialize MLflow tracking
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri("https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow")
mlflow.set_experiment("meta_model_selection")

Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/4f49cb14211d45efabc66f31d11523fb', creation_time=1772020661839, experiment_id='10', last_update_time=1772020661839, lifecycle_stage='active', name='meta_model_selection', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [ ]:
df = pd.read_csv("twitter_cleaned.csv")

# Basic cleaning
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

# Encoding Target
label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

In [ ]:
# Split Data (80/20)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

# Convert targets to numpy arrays for easier indexing in CV loops
y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Training shape: {X_train_text.shape}, Test shape: {X_test_text.shape}")

Training shape: (41292,), Test shape: (10323,)


In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

# Fit on Train, Transform Test
X_train = vectorizer.fit_transform(X_train_text).astype(np.float32)
X_test = vectorizer.transform(X_test_text).astype(np.float32)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

def build_base_models():

    lgbm = LGBMClassifier(
        objective="multiclass",
        num_class=len(label_encoder.classes_),
        n_estimators=794,
        learning_rate=0.19417245978419456,
        max_depth=15,
        num_leaves=179,
        min_child_samples=5,
        subsample=0.9766035386994655,
        colsample_bytree=0.6215441014568336,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        verbose=-1
    )

    rf = RandomForestClassifier(
        n_estimators=400,
        max_features="log2",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    return rf, lgbm

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_base, lgbm_base = build_base_models()

n_classes = len(np.unique(y_train))
n_models = 2 # LogReg + LGBM

# Matrices to store the new "meta-features"
# Shape: (Rows, Models * Classes)
oof_meta = np.zeros((X_train.shape[0], n_classes * n_models))
test_meta = np.zeros((X_test.shape[0], n_classes * n_models))

print("Starting OOF Prediction Loop...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    # Split data for this fold
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr = y_train[train_idx]

    # Clone models to ensure fresh training
    rf = clone(rf_base)
    lgbm = clone(lgbm_base)

    # Train
    rf.fit(X_tr, y_tr)
    lgbm.fit(X_tr, y_tr)

    # 1. Predict on Validation Fold (OOF)
    rf_val_proba = rf.predict_proba(X_val)
    lgbm_val_proba = lgbm.predict_proba(X_val)

    # Fill the OOF matrix
    # Columns 0-2: LogReg probs, Columns 3-5: LGBM probs (assuming 3 classes)
    oof_meta[val_idx] = np.hstack([rf_val_proba, lgbm_val_proba])

    # 2. Predict on Test Set (Accumulate to average later)
    rf_test_proba = rf.predict_proba(X_test)
    lgbm_test_proba = lgbm.predict_proba(X_test)

    test_meta += np.hstack([rf_test_proba, lgbm_test_proba])

# Average the test predictions across folds
test_meta /= 5

print("OOF Generation Complete.")
print(f"Meta-Feature Shape: {oof_meta.shape}")

Starting OOF Prediction Loop...
OOF Generation Complete.
Meta-Feature Shape: (41292, 6)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def objective(trial):
    # 1. Suggest Meta-Learner Type
    meta_type = trial.suggest_categorical("meta_model", ["logistic", "lightgbm", "knn"])

    if meta_type == "logistic":
        model = LogisticRegression(
            C=trial.suggest_float("meta_C", 1e-3, 10.0, log=True),
            solver="lbfgs",
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )

    elif meta_type == "lightgbm":
        model = LGBMClassifier(
            n_estimators=trial.suggest_int("meta_n_estimators", 50, 200),
            learning_rate=trial.suggest_float("meta_lr", 0.01, 0.2),
            num_leaves=trial.suggest_int("meta_num_leaves", 2, 8),
            max_depth=trial.suggest_int("meta_max_depth", 2, 4),
            class_weight="balanced",
            verbosity=-1,
            random_state=42
        )

    else: # KNN
        # KNN requires scaling to work well on probability features
        knn = KNeighborsClassifier(
            n_neighbors=trial.suggest_int("meta_n_neighbors", 5, 50),
            weights=trial.suggest_categorical("meta_weights", ["uniform", "distance"]),
            p=trial.suggest_int("meta_p", 1, 2) # 1=Manhattan, 2=Euclidean
        )
        # Wrap in pipeline to scale data automatically during CV
        model = make_pipeline(StandardScaler(), knn)

    # 2. Evaluation using Cross-Validation
    # This splits the OOF data into 3 folds to test generalization
    cv_scores = cross_val_score(
        model,
        oof_meta,
        y_train,
        cv=3,
        scoring="f1_macro",
        n_jobs=-1
    )

    macro_f1 = cv_scores.mean()

    return macro_f1

In [ ]:
# 1. Run Optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200) # Increased trials since search space is larger

print("Best CV Score:", study.best_value)
print("Best Params:", study.best_params)

# 2. Rebuild Best Model
best_params = study.best_params
model_type = best_params["meta_model"]

if model_type == "logistic":
    final_meta = LogisticRegression(
        C=best_params["meta_C"],
        solver="lbfgs",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
elif model_type == "lightgbm":
    final_meta = LGBMClassifier(
        n_estimators=best_params["meta_n_estimators"],
        learning_rate=best_params["meta_lr"],
        num_leaves=best_params["meta_num_leaves"],
        max_depth=best_params["meta_max_depth"],
        class_weight="balanced",
        verbosity=-1,
        random_state=42
    )
else: # KNN
    knn = KNeighborsClassifier(
        n_neighbors=best_params["meta_n_neighbors"],
        weights=best_params["meta_weights"],
        p=best_params["meta_p"]
    )
    final_meta = make_pipeline(StandardScaler(), knn)

# 3. Fit on ALL OOF Data
final_meta.fit(oof_meta, y_train)

# 4. Predict on Test Meta-Features
test_preds = final_meta.predict(test_meta)

# 5. Final Evaluation
test_f1 = f1_score(y_test, test_preds, average="macro")
test_acc = accuracy_score(y_test, test_preds)

print(f"\nFinal Test Macro F1: {test_f1:.4f}")
print(f"Final Test Accuracy: {test_acc:.4f}")

# Log final model details
with mlflow.start_run(run_name="Best_Meta_Model_With_KNN"):
    mlflow.log_params(best_params)
    mlflow.log_metric("test_f1", test_f1)
    mlflow.sklearn.log_model(final_meta, "model")

[I 2026-02-26 05:21:26,952] A new study created in memory with name: no-name-eb580e93-01db-41af-82f8-c49619e623d5
[I 2026-02-26 05:21:32,621] Trial 0 finished with value: 0.9143925990082115 and parameters: {'meta_model': 'lightgbm', 'meta_n_estimators': 86, 'meta_lr': 0.19789733509013094, 'meta_num_leaves': 7, 'meta_max_depth': 2}. Best is trial 0 with value: 0.9143925990082115.
[I 2026-02-26 05:21:34,523] Trial 1 finished with value: 0.9130865738427091 and parameters: {'meta_model': 'knn', 'meta_n_neighbors': 27, 'meta_weights': 'uniform', 'meta_p': 2}. Best is trial 0 with value: 0.9143925990082115.
[I 2026-02-26 05:21:36,581] Trial 2 finished with value: 0.9143811732396007 and parameters: {'meta_model': 'lightgbm', 'meta_n_estimators': 80, 'meta_lr': 0.08254618038577885, 'meta_num_leaves': 6, 'meta_max_depth': 3}. Best is trial 0 with value: 0.9143925990082115.
[I 2026-02-26 05:21:38,881] Trial 3 finished with value: 0.914293209645371 and parameters: {'meta_model': 'lightgbm', 'meta

Best CV Score: 0.9150965891709238
Best Params: {'meta_model': 'lightgbm', 'meta_n_estimators': 72, 'meta_lr': 0.12794411234826156, 'meta_num_leaves': 8, 'meta_max_depth': 4}

Final Test Macro F1: 0.9297
Final Test Accuracy: 0.9303


2026/02/26 05:27:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/26 05:27:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_Meta_Model_With_KNN at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/10/runs/f010d2f175214dddbca54c000ce98b08
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/10
